# Late Chunking: Practical Demo

In this notebook we demonstrate **Late Chunking** on a real Wikipedia article, using a real embedding model and a real LLM (Groq).

**Normal chunking** → split the document into chunks, then embed each chunk *independently* (chunks lose the context of each other).

**Late chunking** → pass the *whole document* through the embedding model to get token-level embeddings, *then* chunk those embeddings by averaging the tokens of each chunk.

```
Normal : document → split into chunks → embed each chunk alone
Late   : document → embed WHOLE document (token embeddings) → split → average tokens per chunk
```

**What we will do**
1. Setup (LLM + long-context embedding model)
2. Load a real document
3. Chunk it into sentences and see which chunks lose meaning when read alone
4. Normal chunking embeddings (baseline)
5. Late chunking embeddings
6. Compare both on the same question
7. Full RAG (retrieve → augment → generate) with both
8. Summary: advantages, disadvantages, scenarios

## 1. Setup

### 1.1 Install and connect the LLM

In [1]:
pip install langchain-groq -q # q for silent installation

In [2]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

### 1.2 Load the embedding model

Late chunking needs an embedding model that:
- **supports long context** (the whole document must fit in one pass),
- gives us **token-level embeddings** (so we can chunk *after* embedding),
- is trained with **mean pooling** (so averaging tokens gives a valid chunk embedding).

We use `nomic-ai/modernbert-embed-base` (8192 tokens, mean pooling). It loads with plain `transformers`, no custom code needed. Any model with these three properties works (the notes mention Jina v2).

> Tip: this runs fine on CPU for this small document, but a GPU runtime is faster.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = 'nomic-ai/modernbert-embed-base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()  # eval = inference mode

print('Running on:', device)

Running on: cpu


## 2. Load a real document

We fetch the introduction of the Wikipedia article on **Berlin**. Intro paragraphs like this one are full of sentences such as *"Its population is..."* or *"It is..."*, which only make sense with the sentences around them.

In [5]:
import requests

# Wikipedia API: plain text of the intro section of an article
url = 'https://en.wikipedia.org/w/api.php'
params = {
    'action': 'query',
    'titles': 'Berlin',
    'prop': 'extracts',
    'exintro': 1,        # only the intro section
    'explaintext': 1,    # plain text, no HTML
    'format': 'json'
}
headers = {'User-Agent': 'late-chunking-demo/1.0'}  # Wikipedia asks clients to send a User-Agent

data = requests.get(url, params=params, headers=headers).json()
page = next(iter(data['query']['pages'].values()))
text = page['extract']

print('Words:', len(text.split()))
print()
print(text)

Words: 506

Berlin is the capital and largest city of Germany. With 3.7 million inhabitants, it has the highest population within its city limits of any city in the European Union (EU). The metropolis is also one of the states of Germany, being the third-smallest state in the country by area. Berlin is surrounded by the state of Brandenburg, bordering Brandenburg's capital Potsdam to the southwest. The urban area of Berlin has a population of 4.5 million, making it the second most populous in Germany after the Ruhr. The Berlin-Brandenburg capital region has around 6.2 million inhabitants and is Germany's second-largest metropolitan region after the Rhine-Ruhr region, as well as the fifth-biggest metropolitan region by GDP in the EU. Berlin is a global city of culture, politics, media and science.
Berlin was built along the banks of the Spree river, which flows into the Havel in the western borough of Spandau. First documented in the 13th century and at the crossing of two important his

## 3. Chunk the document

Late chunking works with **any chunking strategy**. All it needs is the *boundaries* of each chunk. Here we use one sentence per chunk, which makes the effect easy to see.

We keep the character position (start, end) of every chunk, because later we will use these positions to pick the right tokens.

In [6]:
from nltk.tokenize import PunktSentenceTokenizer

# span_tokenize gives (start, end) character positions of every sentence
spans = list(PunktSentenceTokenizer().span_tokenize(text))
chunks = [text[start:end] for start, end in spans]

print('Number of chunks:', len(chunks))
for i, chunk in enumerate(chunks):
    print(i, '|', chunk[:90])

Number of chunks: 27
0 | Berlin is the capital and largest city of Germany.
1 | With 3.7 million inhabitants, it has the highest population within its city limits of any 
2 | The metropolis is also one of the states of Germany, being the third-smallest state in the
3 | Berlin is surrounded by the state of Brandenburg, bordering Brandenburg's capital Potsdam 
4 | The urban area of Berlin has a population of 4.5 million, making it the second most populo
5 | The Berlin-Brandenburg capital region has around 6.2 million inhabitants and is Germany's 
6 | Berlin is a global city of culture, politics, media and science.
7 | Berlin was built along the banks of the Spree river, which flows into the Havel in the wes
8 | First documented in the 13th century and at the crossing of two important historic trade r
9 | Berlin served as a scientific, artistic, and philosophical hub during the Age of Enlighten
10 | During the Gründerzeit, an industrialization-induced economic boom triggered a rapid popul

### Chunks that lose meaning when read alone

Chunks that start with a pronoun (*It, Its, They, ...*) refer to something mentioned in an earlier chunk. In normal chunking, the embedding of such a chunk **cannot know** what "it" is.

In [7]:
pronouns = ('It', 'Its', 'They', 'Their', 'This', 'These', 'He', 'She')

for i, chunk in enumerate(chunks):
    if chunk.split()[0] in pronouns:
        print(i, '|', chunk[:120])

16 | Its economy is based on high tech and the service sector, encompassing a diverse range of creative industries, startup c


## 4. Normal chunking (baseline)

Each chunk goes through the embedding model **alone**. First we write a small helper that turns texts into embeddings with **mean pooling** (average of all token embeddings).

This model expects a prefix: `search_document: ` for documents and `search_query: ` for questions.

In [8]:
def embed_texts(texts):
    # Tokenize (padding makes all texts in the batch the same length)
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=8192,
                       return_tensors='pt').to(device)

    # One embedding per TOKEN: shape (batch, tokens, 768)
    with torch.no_grad():
        token_embs = model(**inputs).last_hidden_state

    # Mean pooling: average the tokens (ignore padding tokens using the mask)
    mask = inputs['attention_mask'].unsqueeze(-1)
    pooled = (token_embs * mask).sum(dim=1) / mask.sum(dim=1)

    # Normalize so that dot product = cosine similarity
    return torch.nn.functional.normalize(pooled, dim=1)

In [9]:
doc_prefix = 'search_document: '

# Every chunk is embedded independently -> no knowledge of the other chunks
normal_embs = embed_texts([doc_prefix + chunk for chunk in chunks])

print('Normal chunk embeddings:', normal_embs.shape)  # (number of chunks, 768)

Normal chunk embeddings: torch.Size([27, 768])


## 5. Late chunking

Three steps:
1. Pass the **whole document** through the model once → one embedding per token. Because of attention, every token embedding is influenced by the entire document.
2. Find which tokens belong to each chunk (using the character positions from Section 3).
3. **Average** the token embeddings of each chunk → the chunk embedding.

### Step 1: Embed the whole document (token level)

In [10]:
full_text = doc_prefix + text
prefix_len = len(doc_prefix)   # chunk positions must be shifted by the prefix length

inputs = tokenizer(full_text, return_offsets_mapping=True, truncation=True,
                   max_length=8192, return_tensors='pt')

# offsets: (start, end) character position of every token in full_text
offsets = inputs.pop('offset_mapping')[0]
print('Total tokens in the document:', inputs['input_ids'].shape[1], '(model limit: 8192)')

with torch.no_grad():
    token_embs = model(**inputs.to(device)).last_hidden_state[0]  # (tokens, 768)

print('Token embeddings:', token_embs.shape)

Total tokens in the document: 709 (model limit: 8192)
Token embeddings: torch.Size([709, 768])


### Step 2 and 3: Pick the tokens of each chunk and average them

In [11]:
late_embs = []

for start, end in spans:
    # Shift positions because we added a prefix in front of the text
    start, end = start + prefix_len, end + prefix_len

    # True for tokens that overlap this chunk (special tokens have (0, 0) offsets, so they are skipped)
    in_chunk = (offsets[:, 1] > start) & (offsets[:, 0] < end) & (offsets[:, 1] > offsets[:, 0])

    # Average the token embeddings of this chunk -> chunk embedding
    late_embs.append(token_embs[in_chunk.to(device)].mean(dim=0))

late_embs = torch.nn.functional.normalize(torch.stack(late_embs), dim=1)

print('Late chunk embeddings:', late_embs.shape)  # same shape as normal_embs

Late chunk embeddings: torch.Size([27, 768])


## 6. Compare: same question, two kinds of chunk embeddings

We embed one question and compute its cosine similarity with every chunk, once using normal embeddings and once using late embeddings.

Scores of the two methods come from different embeddings, so compare the **rank** (1 = best match) rather than only the raw score.

In [12]:
import pandas as pd

query = 'What is the population of Berlin?'
query_emb = embed_texts(['search_query: ' + query])[0]

# Dot product of normalized vectors = cosine similarity
normal_scores = (normal_embs @ query_emb).cpu().numpy()
late_scores = (late_embs @ query_emb).cpu().numpy()

df = pd.DataFrame({
    'chunk': [chunk[:80] + '...' for chunk in chunks],
    'normal_score': normal_scores.round(3),
    'late_score': late_scores.round(3),
})

# Rank 1 = most similar chunk to the question
df['normal_rank'] = df['normal_score'].rank(ascending=False).astype(int)
df['late_rank'] = df['late_score'].rank(ascending=False).astype(int)

# Positive = the chunk moved UP in the ranking with late chunking
df['rank_change'] = df['normal_rank'] - df['late_rank']

pd.set_option('display.max_colwidth', 90)
df.sort_values('rank_change', ascending=False).head(10)

,chunk,normal_score,late_score,normal_rank,late_rank,rank_change
2,"The metropolis is also one of the states of Germany, being the third-smallest st...",0.392,0.706,23,4,19
26,"About 1/3 of the city's area is composed of forests, parks and gardens, rivers, ...",0.234,0.641,26,11,15
23,"Other landmarks include the Brandenburg Gate, the Reichstag building, Potsdamer ...",0.362,0.634,24,13,11
1,"With 3.7 million inhabitants, it has the highest population within its city limi...",0.492,0.776,11,2,9
25,"The city includes lakes in the western and southeastern boroughs, the largest of...",0.263,0.621,25,17,8
8,First documented in the 13th century and at the crossing of two important histor...,0.467,0.655,13,7,6
12,"After World War II and following Berlin's occupation, the city was split into We...",0.459,0.643,15,10,5
13,"East Berlin was declared the capital of East Germany, while Bonn became the West...",0.416,0.625,21,16,5
14,"Following German reunification in 1990, Berlin once again became the capital of ...",0.452,0.634,16,13,3
16,"Its economy is based on high tech and the service sector, encompassing a diverse...",0.135,0.593,27,25,2


**How to read this table:** look at chunks with a high `rank_change`. Those are chunks that late chunking moved up for this question, typically sentences that depend on surrounding sentences (pronouns, "the city", "the capital", ...).

## 7. Full RAG: Retrieve → Augment → Generate

Now we use both kinds of embeddings in a real RAG flow with the same question and the same LLM, and compare the retrieved chunks and the answers.

In [13]:
def ask(question, retrieved_chunks):
    # Augmentation: combine the retrieved chunks with the user question
    context = '\n'.join(retrieved_chunks)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}"""

    # Generation: LLM answers on the basis of the retrieved info
    return llm.invoke(prompt).content

In [14]:
top_k = 2

for method, scores in [('Normal chunking', normal_scores), ('Late chunking', late_scores)]:
    # Retrieval: indices of the top_k most similar chunks
    top_ids = scores.argsort()[::-1][:top_k]
    retrieved = [chunks[i] for i in top_ids]

    print('=' * 60)
    print(method)
    print('=' * 60)
    print('Retrieved chunks:')
    for i in top_ids:
        print(f'  [{i}]', chunks[i])
    print()
    print('LLM answer:', ask(query, retrieved))
    print()

Normal chunking
Retrieved chunks:
  [4] The urban area of Berlin has a population of 4.5 million, making it the second most populous in Germany after the Ruhr.
  [5] The Berlin-Brandenburg capital region has around 6.2 million inhabitants and is Germany's second-largest metropolitan region after the Rhine-Ruhr region, as well as the fifth-biggest metropolitan region by GDP in the EU.

LLM answer: 4.5 million.

Late chunking
Retrieved chunks:
  [4] The urban area of Berlin has a population of 4.5 million, making it the second most populous in Germany after the Ruhr.
  [1] With 3.7 million inhabitants, it has the highest population within its city limits of any city in the European Union (EU).

LLM answer: The population of Berlin is 3.7 million inhabitants.



## 8. Summary

### Why late chunking can be better
Each token embedding is influenced by the **entire document** (via attention). When we average tokens into a chunk embedding, that embedding "knows about" the rest of the document, so chunks like *"Its population is..."* still carry the meaning of the sentences around them.

### Disadvantages (the catch)
- Needs a **long-context** embedding model, and the whole document must fit in its context window (8192 tokens here). Longer documents have to be split into large parts first.
- Needs **token-level embeddings**. APIs that return only one pooled vector per text (like the standard OpenAI embeddings API) cannot be used.
- The model must be trained with **mean pooling**.
- More compute: the whole document goes through the model instead of small independent chunks.

### When to use it
- Documents where chunks refer back to earlier text (pronouns, "the company", "this method", "the city").
- Wikipedia-style articles, reports, legal and technical documents.
- When you already have a good chunking strategy (sentence, recursive, semantic) and want better embeddings **for the same chunks**.

### When not to use it
- Chunks are already self-contained (FAQs, independent short records).
- Documents far longer than the model's context window.
- You must use an embedding API that gives only a single vector per text.